**Problema de Negócio:** Transportadoras cobram o frete baseando-se no maior valor entre o peso físico da carga e o peso cubado (espaço ocupado). Devido a processos manuais, faturas frequentemente chegam com cálculos de cubagem errados, tarifas acima do contrato ou moedas misturadas (BRL e USD), causando vazamento financeiro.

**Objetivo:** Cruzar as faturas brutas recebidas com o tarifário oficial da empresa. Calcular o peso taxável real (fator de cubagem de 6000), padronizar moedas e identificar exatamente quantos reais a empresa pagou a mais por erros das transportadoras.

In [0]:
%python
import pandas as pd

tarifas = pd.DataFrame({
    'carrier': ['Expresso Sul', 'Norte Log', 'Global Freight'],
    'tarifa_por_kg': [15.50, 12.00, 25.00]
})

faturas = pd.DataFrame({
    'invoice_id': ['INV-001', 'INV-002', 'INV-003', 'INV-004', 'INV-005'],
    'carrier': ['Expresso Sul', 'Norte Log', 'Global Freight', 'Expresso Sul', 'Global Freight'],
    'peso_fisico_kg': [10.0, 50.0, 5.0, 20.0, 8.0],
    'comp_cm': [40, 100, 30, 50, 60],
    'larg_cm': [30, 80, 20, 50, 40],
    'alt_cm': [20, 80, 20, 50, 40],
    'valor_cobrado': [155.00, 1500.00, 50.00, 400.00, 300.00], 
    'moeda': ['BRL', 'BRL', 'USD', 'BRL', 'USD'] 
})

spark.createDataFrame(faturas).createOrReplaceTempView("bronze_faturas")
spark.createDataFrame(tarifas).createOrReplaceTempView("bronze_tarifario")

display(spark.table("bronze_faturas"))

invoice_id,carrier,peso_fisico_kg,comp_cm,larg_cm,alt_cm,valor_cobrado,moeda
INV-001,Expresso Sul,10.0,40,30,20,155.0,BRL
INV-002,Norte Log,50.0,100,80,80,1500.0,BRL
INV-003,Global Freight,5.0,30,20,20,50.0,USD
INV-004,Expresso Sul,20.0,50,50,50,400.0,BRL
INV-005,Global Freight,8.0,60,40,40,300.0,USD


In [0]:
%python
from pyspark.sql import functions as F

df_bronze = spark.table("bronze_faturas")

df_silver = df_bronze.withColumn("peso_cubado_kg", (F.col("comp_cm") * F.col("larg_cm") * F.col("alt_cm")) / 6000)

df_silver = df_silver.withColumn("peso_taxavel_kg", F.greatest(F.col("peso_fisico_kg"), F.col("peso_cubado_kg")))

df_silver = df_silver.withColumn("valor_cobrado_brl", 
    F.when(F.col("moeda") == 'USD', F.col("valor_cobrado") * 5.0)
    .otherwise(F.col("valor_cobrado"))
)

df_silver.createOrReplaceTempView("silver_faturas")

display(df_silver.select("invoice_id", "peso_fisico_kg", "peso_cubado_kg", "peso_taxavel_kg", "valor_cobrado_brl"))

invoice_id,peso_fisico_kg,peso_cubado_kg,peso_taxavel_kg,valor_cobrado_brl
INV-001,10.0,4.0,10.0,155.0
INV-002,50.0,106.66666666666667,106.66666666666667,1500.0
INV-003,5.0,2.0,5.0,250.0
INV-004,20.0,20.833333333333332,20.833333333333332,400.0
INV-005,8.0,16.0,16.0,1500.0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold_auditoria_frete AS
SELECT 
    f.invoice_id,
    f.carrier,
    ROUND(f.peso_taxavel_kg, 2) AS peso_final_cobrado_kg,
    t.tarifa_por_kg AS tarifa_contrato_brl,
    ROUND(f.peso_taxavel_kg * t.tarifa_por_kg, 2) AS frete_correto_brl,
    ROUND(f.valor_cobrado_brl, 2) AS frete_cobrado_brl,
    ROUND(f.valor_cobrado_brl - (f.peso_taxavel_kg * t.tarifa_por_kg), 2) AS prejuizo_identificado_brl
FROM silver_faturas f
JOIN bronze_tarifario t ON f.carrier = t.carrier;

SELECT * FROM gold_auditoria_frete ORDER BY prejuizo_identificado_brl DESC;

invoice_id,carrier,peso_final_cobrado_kg,tarifa_contrato_brl,frete_correto_brl,frete_cobrado_brl,prejuizo_identificado_brl
INV-005,Global Freight,16.0,25.0,400.0,1500.0,1100.0
INV-002,Norte Log,106.67,12.0,1280.0,1500.0,220.0
INV-003,Global Freight,5.0,25.0,125.0,250.0,125.0
INV-004,Expresso Sul,20.83,15.5,322.92,400.0,77.08
INV-001,Expresso Sul,10.0,15.5,155.0,155.0,0.0


In [0]:
%sql
SELECT 
    carrier, 
    SUM(prejuizo_identificado_brl) AS total_recuperavel_brl
FROM gold_auditoria_frete
WHERE prejuizo_identificado_brl > 0
GROUP BY carrier
ORDER BY total_recuperavel_brl DESC;

carrier,total_recuperavel_brl
Global Freight,1225.0
Norte Log,220.0
Expresso Sul,77.08


Databricks visualization. Run in Databricks to view.